# D127 — Shopping Cart Testing with pytest

This notebook tests the same shopping-cart behavior introduced in **D124**, but uses `pytest` instead of Python's `unittest` framework.

## Install pytest

In a terminal:

```bash
python -m pip install pytest
```

Or in a notebook kernel:

```python
%pip install pytest
```

## Business rules from D124

- A `CartItem` has an ID, name, quantity, unit price, and calculated amount.
- A `Cart` stores zero or more items.
- `items_count` is the sum of item quantities.
- `amount` is the sum of item amounts.
- Adding, removing, emptying, or updating the cart refreshes both totals.

## Shopping-cart code under test

In a real project, place these classes in a source file such as `src/shop/cart.py`.

In [ ]:
class CartItem:
    def __init__(self, item_id, name, qty, unit_price):
        if qty <= 0:
            raise ValueError("Quantity must be positive")
        if unit_price < 0:
            raise ValueError("Unit price cannot be negative")
        self.id = item_id
        self.name = name
        self.qty = qty
        self.unit_price = unit_price

    @property
    def amount(self):
        return self.qty * self.unit_price


class Cart:
    def __init__(self):
        self.initialize_cart()

    def initialize_cart(self):
        self.items = {}
        self.items_count = 0
        self.amount = 0

    def calculate_cart_total(self):
        self.items_count = sum(item.qty for item in self.items.values())
        self.amount = sum(item.amount for item in self.items.values())
        return self.amount

    def add_item_to_cart(self, item):
        if not isinstance(item, CartItem):
            raise TypeError("item must be a CartItem")
        if item.id in self.items:
            self.items[item.id].qty += item.qty
        else:
            self.items[item.id] = item
        self.calculate_cart_total()

    def remove_item_from_cart(self, item_id):
        if item_id not in self.items:
            raise KeyError("Item not found")
        removed_item = self.items.pop(item_id)
        self.calculate_cart_total()
        return removed_item

    def empty_cart(self):
        self.items.clear()
        self.calculate_cart_total()

    def update_cart(self, item_id, qty=None, unit_price=None):
        if item_id not in self.items:
            raise KeyError("Item not found")
        if qty is None and unit_price is None:
            raise ValueError("Provide quantity or unit price")
        if qty is not None and qty <= 0:
            raise ValueError("Quantity must be positive")
        if unit_price is not None and unit_price < 0:
            raise ValueError("Unit price cannot be negative")
        item = self.items[item_id]
        item.qty = qty if qty is not None else item.qty
        item.unit_price = (
            unit_price if unit_price is not None else item.unit_price
        )
        self.calculate_cart_total()

## `unittest` and pytest comparison

| D124 with `unittest` | D127 with pytest |
|---|---|
| Inherit from `unittest.TestCase` | Write plain functions named `test_*` |
| `self.assertEqual(actual, expected)` | `assert actual == expected` |
| `self.assertIn(value, collection)` | `assert value in collection` |
| `self.assertRaises(ValueError)` | `pytest.raises(ValueError)` |
| `setUp()` | A fixture decorated with `@pytest.fixture` |
| Build a `TestSuite` manually | pytest discovers tests automatically |

## Plain test functions and normal assertions

pytest does not require a test class. The function name must begin with `test_`.

In [ ]:
def test_item_amount_is_quantity_times_price():
    item = CartItem(1, "Keyboard", 2, 1500)

    assert item.amount == 3000


def test_new_cart_is_empty():
    cart = Cart()

    assert cart.items == {}
    assert cart.items_count == 0
    assert cart.amount == 0

## Fixtures replace repeated `setUp` code

A fixture prepares fresh test data. pytest passes it into any test that has a parameter with the fixture's name. By default, `cart_with_keyboard` runs separately for every test, so tests do not share mutated state.

In [ ]:
import pytest


@pytest.fixture
def cart_with_keyboard():
    cart = Cart()
    cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))
    return cart


def test_add_new_item_updates_state():
    cart = Cart()
    cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))

    assert cart.items_count == 2
    assert cart.amount == 3000
    assert 1 in cart.items


def test_same_item_increases_quantity(cart_with_keyboard):
    cart_with_keyboard.add_item_to_cart(
        CartItem(1, "Keyboard", 1, 1500)
    )

    assert cart_with_keyboard.items[1].qty == 3
    assert cart_with_keyboard.items_count == 3
    assert cart_with_keyboard.amount == 4500

## Test remove, empty, and update behavior

Each test concentrates on one behavior and checks the resulting cart state.

In [ ]:
def test_remove_existing_item(cart_with_keyboard):
    removed = cart_with_keyboard.remove_item_from_cart(1)

    assert removed.name == "Keyboard"
    assert cart_with_keyboard.items_count == 0
    assert cart_with_keyboard.amount == 0


def test_empty_cart_resets_all_state(cart_with_keyboard):
    cart_with_keyboard.add_item_to_cart(CartItem(2, "Mouse", 1, 500))
    cart_with_keyboard.empty_cart()

    assert cart_with_keyboard.items == {}
    assert cart_with_keyboard.items_count == 0
    assert cart_with_keyboard.amount == 0


def test_update_quantity_and_price(cart_with_keyboard):
    cart_with_keyboard.update_cart(1, qty=1, unit_price=1000)

    assert cart_with_keyboard.items[1].amount == 1000
    assert cart_with_keyboard.items_count == 1
    assert cart_with_keyboard.amount == 1000

## Test exceptions with `pytest.raises`

The test passes only when the expected exception is raised inside the context block. The optional `match` argument also checks part of the error message.

In [ ]:
def test_zero_quantity_is_rejected():
    with pytest.raises(ValueError, match="Quantity must be positive"):
        CartItem(1, "Keyboard", 0, 1500)


def test_unknown_item_raises_key_error():
    cart = Cart()

    with pytest.raises(KeyError, match="Item not found"):
        cart.remove_item_from_cart(99)


def test_add_rejects_non_cart_item():
    cart = Cart()

    with pytest.raises(TypeError, match="item must be a CartItem"):
        cart.add_item_to_cart("Keyboard")

## Check several inputs with parametrization

`@pytest.mark.parametrize` runs one test once for every data row. This avoids repeating nearly identical tests.

In [ ]:
@pytest.mark.parametrize(
    "qty, unit_price, expected_amount",
    [
        (1, 1500, 1500),
        (2, 1500, 3000),
        (3, 500, 1500),
    ],
)
def test_cart_item_amount(qty, unit_price, expected_amount):
    item = CartItem(1, "Keyboard", qty, unit_price)

    assert item.amount == expected_amount

## Recommended files

pytest is normally run against `.py` files rather than test functions stored only in a notebook:

```text
shopping-cart/
├── src/
│   └── shop/
│       ├── __init__.py
│       └── cart.py
└── tests/
    ├── conftest.py
    └── test_cart.py
```

- Put `Cart` and `CartItem` in `src/shop/cart.py`.
- Put test functions in `tests/test_cart.py`.
- Put fixtures shared by multiple test files in `tests/conftest.py`.
- Import with `from shop.cart import Cart, CartItem`.

## Run the tests

From the project root:

```bash
# Run all discovered tests
python -m pytest

# Show individual test names
python -m pytest -v

# Run the shopping-cart test file
python -m pytest tests/test_cart.py

# Run tests containing "update" in their name
python -m pytest -k update
```

No manual suite is necessary. pytest discovers files named `test_*.py` and functions named `test_*`.

## Notebook smoke test

The pytest functions above are collected when placed in a test file. This final direct check confirms that the notebook's shopping-cart code also works while reading the lesson.

In [ ]:
cart = Cart()
cart.add_item_to_cart(CartItem(1, "Keyboard", 2, 1500))
cart.add_item_to_cart(CartItem(2, "Mouse", 1, 500))
assert cart.items_count == 3
assert cart.amount == 3500

cart.update_cart(1, qty=1)
assert cart.items_count == 2
assert cart.amount == 2000
print("Shopping-cart smoke test passed")

## Key lessons

- pytest tests are plain functions with normal `assert` statements.
- Fixtures provide fresh reusable setup data.
- `pytest.raises` verifies expected errors.
- Parametrization checks several input combinations without duplicated code.
- pytest discovers the test suite automatically from standard names.